<a href="https://colab.research.google.com/github/machancejoy-max/colab-git-demo-JOY/blob/main/Final_project_PAAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Download the Sentiment140 ZIP file
!wget https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip

# Unzip the dataset
!unzip trainingandtestdata.zip


--2026-05-13 17:22:54--  https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 81363704 (78M) [application/zip]
Saving to: ‘trainingandtestdata.zip.1’

trainingandtestdata 100%[===================>]  77.59M  12.0MB/s    in 6.3s    

2026-05-13 17:23:00 (12.3 MB/s) - ‘trainingandtestdata.zip.1’ saved [81363704/81363704]

Archive:  trainingandtestdata.zip
replace testdata.manual.2009.06.14.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import pandas as pd

cols = ["target", "id", "date", "flag", "user", "text"]

df = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    names=cols
)

df.head()


In [ ]:
import re

def clean_tweet(text):
    text = text.lower()  # lowercase
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)  # remove URLs
    text = re.sub(r"@\w+", "", text)  # remove mentions
    text = re.sub(r"#\w+", "", text)  # remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove special characters
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

df["clean_text"] = df["text"].apply(clean_tweet)

df[["text", "clean_text"]].head()


In [ ]:
df["sentiment"] = df["target"].map({
    0: "negative",
    2: "neutral",
    4: "positive"
})

df["sentiment"].value_counts()


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", max_features=5000)
X = vectorizer.fit_transform(df["clean_text"])

X.shape


In [ ]:
from sklearn.model_selection import train_test_split

sample_df = df.sample(200000, random_state=42)  # 200k tweets

X = sample_df["clean_text"]
y = sample_df["sentiment"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X_train_vec, y_train)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_pred = model.predict(X_val_vec)

acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average="weighted")

print("Accuracy:", acc)
print("F1-Score:", f1)
print("\nClassification Report:\n", classification_report(y_val, y_pred))


In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import re

# Load model + vectorizer
model = joblib.load("logistic_model_v1.pkl")
vectorizer = joblib.load("vectorizer_v1.pkl")

app = Flask(__name__)

def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    tweet = data.get("text", "")

    cleaned = clean_tweet(tweet)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]

    return jsonify({"sentiment": pred})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)



In [ ]:
import joblib

joblib.dump(model, "logistic_model_v1.pkl")
joblib.dump(vectorizer, "vectorizer_v1.pkl")




In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY logistic_model_v1.pkl logistic_model.pkl
COPY vectorizer_v1.pkl vectorizer.pkl

EXPOSE 5000

CMD ["python", "app.py"]


In [ ]:
%%writefile requirements.txt
flask
scikit-learn
joblib
numpy
pandas
